# Levantamento do dicionário de dados

Inspeciona nomes, tipos inferidos e categorias da base configurada em `BASE_CSV_PATH`. Não exibe linhas da base nem calcula categorias para identificadores. As contagens são agregadas.

In [1]:
import os
from pathlib import Path
import pandas as pd

base_csv_path = os.getenv('BASE_CSV_PATH')
if not base_csv_path:
    raise EnvironmentError('Defina BASE_CSV_PATH para o CSV da base antes de executar o notebook.')

csv_path = Path(base_csv_path)
if not csv_path.is_file():
    raise FileNotFoundError(f'CSV não encontrado no caminho definido em BASE_CSV_PATH: {csv_path}')

df = pd.read_csv(csv_path, sep=';', encoding='utf-8-sig', decimal=',', low_memory=False)
print(f'Dimensões: {df.shape[0]:,} linhas × {df.shape[1]} colunas')

Dimensões: 59,059 linhas × 66 colunas


## Tipos inferidos e preenchimento

O quadro mostra apenas metadados por coluna. Identificadores `HANDLE*` são listados pelo nome e tipo, sem exibir valores ou cardinalidade.

In [2]:
resumo = pd.DataFrame({
    'tipo_inferido': df.dtypes.astype(str),
    'preenchidos': df.notna().sum(),
    'ausentes': df.isna().sum(),
})
resumo

,tipo_inferido,preenchidos,ausentes
HANDLEANOSEMESTRE,int64,59059,0
ANOSEMESTRE,str,59059,0
HANDLEALUNOANOSEMESTRE,float64,48275,10784
HANDLEALUNO,int64,59059,0
HANDLEALUNOCURRICULO,int64,59059,0
...,...,...,...
IA34_03,float64,44417,14642
IA65_03,float64,44417,14642
ATIVCOMPL_MATRICULADAS,float64,44417,14642
ATIVCOMPL_APROVADAS,float64,44417,14642


## Categorias

`value_counts()` é aplicado somente às colunas categóricas nominais abaixo. Identificadores não entram na lista. Revise a lista se as colunas categóricas do arquivo mudarem.

In [3]:
colunas_categoricas = [
    'ANOSEMESTRE', 'SIGLACURSO', 'TURNO', 'SEXO',
    'MATRICULAEFETIVADA', 'CODIGOSTATUSALUNO', 'STATUSALUNO',
    'INGRESSO', 'ORIGEMREGISTRO',
]
colunas_presentes = [c for c in colunas_categoricas if c in df.columns]
for coluna in colunas_presentes:
    print(f'\n{coluna}')
    display(df[coluna].value_counts(dropna=False).rename_axis('categoria').to_frame('frequencia'))

faltantes = sorted(set(colunas_categoricas) - set(df.columns))
if faltantes:
    print('Colunas categóricas esperadas ausentes:', faltantes)


ANOSEMESTRE


,frequencia
categoria,
2016/1,3449
2014/1,3202
2015/1,2476
2026/1,2453
2013/1,2140
2014/2,2136
2016/2,2121
2015/2,2078
2013/2,2047



SIGLACURSO


,frequencia
categoria,
GEC,13791
GET,12533
GEA,8801
GEB,6240
GES,5947
GEE,4369
TAI,3229
TGT,1811
GEP,1331



TURNO


,frequencia
categoria,
Integral,43825
Noturno,15234


... (8 outputs left out)


INGRESSO

,frequencia
categoria,
2014/1,3944
2012/1,3615
2013/1,3545
2015/1,3368
2011/1,3224
2010/1,2305
2018/1,2253
2019/1,2242
2016/1,2166



ORIGEMREGISTRO


,frequencia
categoria,
ACADEMICA,44417
OCORRENCIA,14642
